### Ingest Result.json file
1. Read the file using spark dataframe reader API
2. Define and enforce schema
3. Add metadata columns
   - Source File
    - ingestion timestamp
3. Write to bronze delta table 

Note - JSON is in multi line format


In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType

results_schema = StructType(fields=[
    StructField("date",DateType()),
    StructField("raceName",StringType()),
    StructField("round",IntegerType()),
    StructField("season",IntegerType()),
    StructField("url",StringType()),
    StructField("constructorId",StringType()),
    StructField("driverId",StringType()),
    StructField("grid",IntegerType()),
    StructField("laps",IntegerType()),
    StructField("number",IntegerType()),
    StructField("points",FloatType()),
    StructField("position",IntegerType()),
    StructField("positionText",StringType()),
    StructField("status",StringType()),
])


In [0]:
sprints_df = spark.read.format('json').schema(results_schema).option('mode','FAILFAST').option('multiLine','true').load(source_file)

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
display(sprints_final_df)

In [0]:
(
    sprints_final_df
    .write
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
select season,count(*) from formula1.bronze.sprints
group by season
order by season 